In [1]:
!pip install torch_geometric
!pip install adamp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for adamp: filename=adamp-0.3.0-py3-none-any.whl size=5984 sha256=73b937a3702d555b000147f89a78a0f02963ec0ef36060f837037adb4201643e
  Stored in directory: /root/.cache/pip/wheels/c7/ad/0f/b41b1c45b18c66e5eef5d2254415af8055c7e2b0934145157d
Successfully built adamp


In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import random

# Set random seeds for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)

# Define the GAT model with pooling and batch normalization
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8, dropout=0.6):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels * heads)

        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=heads)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels * heads)

        self.conv3 = GATConv(hidden_channels * heads, hidden_channels, heads=heads)
        self.bn3 = torch.nn.BatchNorm1d(hidden_channels * heads)

        self.conv4 = GATConv(hidden_channels * heads, hidden_channels, heads=1)
        self.fc = torch.nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

        self._initialize_weights()

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index).relu()
        x = self.bn1(x)  # Batch norm after first conv
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index).relu()
        x = self.bn2(x)  # Batch norm after second conv
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index).relu()
        x = self.bn3(x)  # Batch norm after third conv
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv4(x, edge_index).relu()
        x = global_mean_pool(x, batch)

        x = self.fc(x)
        return F.log_softmax(x, dim=1)

    def _initialize_weights(self):
        # Initialize weights for GATConv and Linear layers
        for module in self.modules():
            if isinstance(module, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    module.bias.data.fill_(0.01)
            elif isinstance(module, GATConv):
                torch.nn.init.xavier_uniform_(module.lin.weight)
                if module.lin.bias is not None:
                    module.lin.bias.data.fill_(0.01)

# Train the model
def train_model(model, loader, val_loader=None, epochs=200, lr=0.001, weight_decay=1e-4, patience=20, grad_clip=2.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    early_stopping_counter = 0
    best_val_loss = float('inf')

    for epoch in range(epochs):
        total_loss = 0
        for data in loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)  # Gradient clipping
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        scheduler.step()

        # Validation step for early stopping
        if val_loader is not None:
            val_loss = evaluate_model_loss(model, val_loader)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                early_stopping_counter = 0
            else:
                early_stopping_counter += 1

            if early_stopping_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    return model

# Evaluate the model loss for validation
def evaluate_model_loss(model, loader):
    model.eval()
    criterion = torch.nn.CrossEntropyLoss()
    total_loss = 0

    with torch.no_grad():
        for data in loader:
            out = model(data)
            loss = criterion(out, data.y)
            total_loss += loss.item()

    return total_loss / len(loader)

# Evaluate the model performance
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            out = model(data)
            preds = out.argmax(dim=1)

            if data.y.ndim == 2:
                labels = data.y.argmax(dim=1)
            else:
                labels = data.y

            all_preds.append(out)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0).view(-1)

    out_prob = F.softmax(all_preds, dim=1).cpu()

    if num_classes == 2:
        auc_roc = roc_auc_score(all_labels.cpu().numpy(), out_prob[:, 1], average='weighted')
    else:
        auc_roc = roc_auc_score(all_labels.cpu().numpy(), out_prob, multi_class='ovr', average='weighted')

    logloss = log_loss(all_labels.cpu().numpy(), out_prob)
    test_acc = (all_preds.argmax(dim=1) == all_labels).sum().item() / len(all_labels)
    f1 = f1_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    precision = precision_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    recall = recall_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')

    return {
        'accuracy': test_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc_roc': auc_roc,
        'log_loss': logloss
    }

# Load and augment datasets
dataset_proteins = TUDataset(root='/content/PROTEINS', name='PROTEINS', transform=NormalizeFeatures())
dataset_mutag = TUDataset(root='/content/MUTAG', name='MUTAG', transform=NormalizeFeatures())

# Split datasets into training and testing sets (70/30)
train_idx_proteins, test_idx_proteins = train_test_split(range(len(dataset_proteins)), test_size=0.3, random_state=42)
train_dataset_proteins = dataset_proteins[train_idx_proteins]
test_dataset_proteins = dataset_proteins[test_idx_proteins]

train_idx_mutag, test_idx_mutag = train_test_split(range(len(dataset_mutag)), test_size=0.3, random_state=42)
train_dataset_mutag = dataset_mutag[train_idx_mutag]
test_dataset_mutag = dataset_mutag[test_idx_mutag]

train_loader_proteins = DataLoader(train_dataset_proteins, batch_size=32, shuffle=True)
test_loader_proteins = DataLoader(test_dataset_proteins, batch_size=32, shuffle=False)

train_loader_mutag = DataLoader(train_dataset_mutag, batch_size=32, shuffle=True)
test_loader_mutag = DataLoader(test_dataset_mutag, batch_size=32, shuffle=False)

# Initialize and train model on PROTEINS
model_proteins = GAT(in_channels=dataset_proteins.num_node_features, hidden_channels=64, out_channels=dataset_proteins.num_classes, dropout=0.5)
train_model(model_proteins, train_loader_proteins, epochs=200)

# Evaluate on PROTEINS dataset
print("\nResults on PROTEINS Dataset Before Transfer Learning:")
proteins_results = evaluate_model(model_proteins, test_loader_proteins, num_classes=dataset_proteins.num_classes)
print(proteins_results)

# Transfer Learning to MUTAG Dataset
model_mutag = GAT(in_channels=dataset_mutag.num_node_features, hidden_channels=64, out_channels=dataset_mutag.num_classes, dropout=0.3)

# Load the pre-trained weights from the PROTEINS model to the MUTAG model
state_dict_proteins = model_proteins.state_dict()
state_dict_mutag = model_mutag.state_dict()

# Freeze conv1 layer for transfer learning
for key in state_dict_proteins:
    if key in state_dict_mutag and state_dict_proteins[key].shape == state_dict_mutag[key].shape:
        state_dict_mutag[key] = state_dict_proteins[key]

model_mutag.load_state_dict(state_dict_mutag, strict=False)

# Train the MUTAG model with the transfer-learned weights
train_model(model_mutag, train_loader_mutag, epochs=200)

# Evaluate on MUTAG dataset
print("\nResults on MUTAG Dataset After Transfer Learning:")
mutag_results = evaluate_model(model_mutag, test_loader_mutag, num_classes=dataset_mutag.num_classes)
print(mutag_results)




Results on PROTEINS Dataset Before Transfer Learning:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


{'accuracy': 0.7155688622754491, 'f1': 0.7169785190077124, 'precision': 0.719017581490593, 'recall': 0.7155688622754491, 'auc_roc': 0.7124732905982906, 'log_loss': 0.7193338266618993}

Results on MUTAG Dataset After Transfer Learning:
{'accuracy': 0.8771929824561403, 'f1': 0.876215856095936, 'precision': 0.8758513931888544, 'recall': 0.8771929824561403, 'auc_roc': 0.8618233618233618, 'log_loss': 0.5173084484330416}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [ ]:
87,